# Link Validation Notebook

This notebook checks every link in the MkDocs `docs/` folder and reports whether links are valid or broken. It focuses on internal Markdown links and URL syntax validation.

In [1]:
import re
from pathlib import Path
import urllib.parse
import requests
import pandas as pd

root = Path('.').resolve()
docs_root = root / 'docs'
md_files = sorted(docs_root.rglob('*.md'))

link_re = re.compile(r'!?\[[^\]]*\]\(([^)]+)\)')
exclude_prefixes = ('mailto:', 'javascript:')


## Load URL List

Collect internal Markdown links from `docs/` and prepare them for validation.

In [2]:
links = []
for md in md_files:
    text = md.read_text(encoding='utf-8', errors='ignore')
    for match in link_re.finditer(text):
        target = match.group(1).strip()
        if not target or target.startswith('#'):
            continue
        links.append({'source': str(md.relative_to(root)), 'target': target})

links_df = pd.DataFrame(links)
links_df.head(20)


,source,target
0,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Lattice%20Type.md
1,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Lattice%20structure.md
2,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Reciprocal%20Lattice.md
3,docs\Condensed Matter Physics.md,Physics%20Note/CMP/X-ray%20Diffraction.md
4,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Bonding.md
5,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Lattice%20vibration.md
6,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Heat%20Capacity.md
7,docs\Condensed Matter Physics.md,Physics%20Note/Debye%20Model%201.md
8,docs\Condensed Matter Physics.md,Physics%20Note/Metal%201.md
9,docs\Condensed Matter Physics.md,Physics%20Note/Metal%20II.md


## Validate URL Format

Check the syntax of each link before attempting network requests.

In [3]:
def normalize_target(target):
    if '#' in target:
        target = target.split('#', 1)[0]
    return target

valid_links = []
for row in links_df.to_dict('records'):
    target = row['target']
    parsed = urllib.parse.urlparse(target)
    if parsed.scheme and parsed.scheme not in ('http', 'https'):
        valid_links.append({**row, 'valid_format': False, 'reason': 'unsupported scheme'})
        continue
    valid_links.append({**row, 'valid_format': True, 'reason': ''})

valid_df = pd.DataFrame(valid_links)
valid_df.head(20)


,source,target,valid_format,reason
0,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Lattice%20Type.md,True,
1,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Lattice%20structure.md,True,
2,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Reciprocal%20Lattice.md,True,
3,docs\Condensed Matter Physics.md,Physics%20Note/CMP/X-ray%20Diffraction.md,True,
4,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Bonding.md,True,
5,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Lattice%20vibration.md,True,
6,docs\Condensed Matter Physics.md,Physics%20Note/CMP/Heat%20Capacity.md,True,
7,docs\Condensed Matter Physics.md,Physics%20Note/Debye%20Model%201.md,True,
8,docs\Condensed Matter Physics.md,Physics%20Note/Metal%201.md,True,
9,docs\Condensed Matter Physics.md,Physics%20Note/Metal%20II.md,True,


## Send HTTP Requests to Each Link

Validate external links by sending HTTP requests and capturing response status codes.

In [4]:
external_links = [row for row in valid_links if urllib.parse.urlparse(row['target']).scheme in ('http', 'https')]
results = []
for row in external_links:
    target = row['target']
    try:
        resp = requests.head(target, allow_redirects=True, timeout=10)
        results.append({**row, 'status_code': resp.status_code, 'reason': ''})
    except Exception as exc:
        results.append({**row, 'status_code': None, 'reason': str(exc)})

results_df = pd.DataFrame(results)
results_df.head(20)


,source,target,valid_format,reason,status_code
0,docs\Physics Note\CMP\Reciprocal Lattice.md,https://www.researchgate.net/post/What-is-the-...,True,,403
1,docs\Physics Note\Master's degree.md,https://www.manchester.ac.uk/study/postgraduat...,True,,200
2,docs\Physics Note\Master's degree.md,https://www.manchester.ac.uk/study/masters/cou...,True,,200
3,docs\Physics Note\Master's degree.md,https://www.dur.ac.uk/departments/academic/phy...,True,,403
4,docs\Physics Note\Master's degree.md,https://www.ox.ac.uk/admissions/graduate/cours...,True,,403
5,docs\Physics Note\Master's degree.md,https://www.ntu.edu.sg/spms/about-us/physics/g...,True,,200
6,docs\Physics Note\RLI Astronomy (Mass of Jupit...,https://doi.org/10.1111/j.1365-2966.2011.18947.x,True,,403


## Report Link Status and Errors

Summarize the validation results and identify broken or unsupported links.

In [8]:
summary = {
    'total_links': len(links_df),
    'invalid_format': sum(1 for row in valid_links if not row['valid_format']),
    'external_links': len(external_links),
    'broken_external': sum(1 for row in results if row['status_code'] not in (None, 200)),
}
summary, results_df[results_df['status_code'] != 200].head(50)


({'total_links': 178,
  'invalid_format': 0,
  'external_links': 7,
  'broken_external': 4},
                                               source  \
 0        docs\Physics Note\CMP\Reciprocal Lattice.md   
 3               docs\Physics Note\Master's degree.md   
 4               docs\Physics Note\Master's degree.md   
 6  docs\Physics Note\RLI Astronomy (Mass of Jupit...   
 
                                               target  valid_format reason  \
 0  https://www.researchgate.net/post/What-is-the-...          True          
 3  https://www.dur.ac.uk/departments/academic/phy...          True          
 4  https://www.ox.ac.uk/admissions/graduate/cours...          True          
 6   https://doi.org/10.1111/j.1365-2966.2011.18947.x          True          
 
    status_code  
 0          403  
 3          403  
 4          403  
 6          403  )

In [9]:
# Direct internal Markdown link validation for docs/
from pathlib import Path
import re

root = Path('.').resolve()
docs_root = root / 'docs'
md_files = sorted(docs_root.rglob('*.md'))
link_re = re.compile(r'!?\[[^\]]*\]\(([^)]+)\)')

broken = []
for md in md_files:
    text = md.read_text(encoding='utf-8', errors='ignore')
    for match in link_re.finditer(text):
        target = match.group(1).strip()
        if not target or target.startswith('#'):
            continue
        if any(target.startswith(p) for p in ('http://', 'https://', 'mailto:', 'javascript:')):
            continue
        if target.startswith('/'):
            broken.append((md.relative_to(root).as_posix(), target, 'absolute path'))
            continue
        target_file = target.split('#', 1)[0]
        if not target_file:
            continue
        target_path = (md.parent / target_file).resolve()
        if target_path.is_dir():
            if not (target_path / 'index.md').exists():
                broken.append((md.relative_to(root).as_posix(), target, 'dir without index'))
            continue
        if not target_path.exists():
            broken.append((md.relative_to(root).as_posix(), target, 'missing file'))

print('Internal docs links checked:', sum(1 for md in md_files for _ in link_re.finditer(md.read_text(encoding="utf-8", errors="ignore"))))
print('Broken internal docs links found:', len(broken))
for md, target, reason in broken[:200]:
    print(f'{md}: {target} -> {reason}')


Internal docs links checked: 178
Broken internal docs links found: 139
docs/Condensed Matter Physics.md: Physics%20Note/CMP/Lattice%20Type.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/CMP/Lattice%20structure.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/CMP/Reciprocal%20Lattice.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/CMP/X-ray%20Diffraction.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/CMP/Bonding.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/CMP/Lattice%20vibration.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/CMP/Heat%20Capacity.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/Debye%20Model%201.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/Metal%201.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/Metal%20II.md -> missing file
docs/Condensed Matter Physics.md: Physics%20Note/Properties%20